In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


df=pd.read_excel('Used_Car_Data.xlsx', sheet_name='Used_Car_Data', header=3)
df.head()

print("Data shape:", df.shape)
df.head(3)

IndentationError: unexpected indent (_reader.py, line 329)

### Question 1

In [4]:
rows, cols = df.shape
print(f"Number of rows: {rows}")
print(f"Number of columns: {cols}")

print("\nTarget-like columns:")
print("- Price Prediction Target: 'Sale_Price_USD'")
print("- Sale Classification Target: 'Sale_Status'")
print("- Sales-Speed Target: 'Days_to_Sell'")


Number of rows: 300
Number of columns: 33

Target-like columns:
- Price Prediction Target: 'Sale_Price_USD'
- Sale Classification Target: 'Sale_Status'
- Sales-Speed Target: 'Days_to_Sell'


### Question 2

In [6]:
missing_count = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_count, 'Percentage': missing_pct}).sort_values(by='Missing_Count', ascending=False)
print("Missing values per column:")
print(missing_df)

print("\nTop 3 columns with highest missingness:")
print(missing_df.head(3))

unsold_count = (df['Sale_Status'] == 'Unsold').sum()
missing_price_unsold = df[df['Sale_Status'] == 'Unsold']['Sale_Price_USD'].isnull().sum()
missing_days_unsold = df[df['Sale_Status'] == 'Unsold']['Days_to_Sell'].isnull().sum()
print(f"\nNumber of Unsold cars: {unsold_count}")
print(f"Missing Sale_Price_USD for Unsold cars: {missing_price_unsold}")
print(f"Missing Days_to_Sell for Unsold cars: {missing_days_unsold}")


Missing values per column:
                              Missing_Count  Percentage
Student_Feature                         300  100.000000
Accident_History                        218   72.666667
Price_per_1000_Miles                     57   19.000000
MinMax_Normalized_Sale_Price             44   14.666667
Sale_Price_USD                           44   14.666667
Days_to_Sell                             44   14.666667
Service_History                          36   12.000000
Mileage_miles                            15    5.000000
Exterior_Color                           10    3.333333
Transmission                              8    2.666667
Asking_Price_USD                          7    2.333333
Listing_ID                                0    0.000000
Vehicle_Type                              0    0.000000
Manufacture_Year                          0    0.000000
Brand                                     0    0.000000
Model                                     0    0.000000
VIN_Code             

### Question 3

In [8]:
cols_to_ignore = ['Listing_ID', 'VIN_Code', 'Photo_URL', 'Upload_Batch', 'Seller_Address']
check_cols = [c for c in df.columns if c not in cols_to_ignore]

duplicates = df[df.duplicated(subset=check_cols, keep=False)]
print(f"Number of near-duplicate rows (ignoring identifiers): {len(duplicates)}")
if len(duplicates) > 0:
    print(duplicates[check_cols + ['Listing_ID', 'VIN_Code']].head(4))


Number of near-duplicate rows (ignoring identifiers): 6
          Brand     Model  ... Listing_ID         VIN_Code
11   Volkswagen    Passat  ...   CAR-0012  VINS5IKEUKGK275
54   Volkswagen      Golf  ...   CAR-0055  VINRAEBQ1PGSX0G
102      Nissan  Frontier  ...   CAR-0103  VINW4DVF278HL74
287  Volkswagen    Passat  ...   CAR-0288  VINS5IKEUKGK275

[4 rows x 30 columns]


### Question 4

In [10]:
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print("Categorical Columns Classifications:")
classifications = {
    'Listing_ID': 'Identifier',
    'VIN_Code': 'Identifier',
    'Brand': 'Nominal (High cardinality)',
    'Model': 'Nominal (High cardinality)',
    'Vehicle_Type': 'Nominal',
    'Fuel_Type': 'Nominal',
    'Transmission': 'Nominal',
    'Exterior_Color': 'Nominal',
    'Condition_Level': 'Ordinal',
    'Accident_History': 'Ordinal / Risk Level',
    'Service_History': 'Ordinal',
    'Seller_Type': 'Nominal',
    'Seller_Address': 'Composite / Text Location',
    'Engine_Spec': 'Composite Technical Field',
    'Sale_Status': 'Binary Outcome',
    'Salesperson_Favorite_Color': 'Irrelevant Noise',
    'Upload_Batch': 'Operational Metadata',
    'Photo_URL': 'Identifier / Irrelevant'
}
for col, cls in classifications.items():
    print(f"- {col}: {cls}")


Categorical Columns Classifications:
- Listing_ID: Identifier
- VIN_Code: Identifier
- Brand: Nominal (High cardinality)
- Model: Nominal (High cardinality)
- Vehicle_Type: Nominal
- Fuel_Type: Nominal
- Transmission: Nominal
- Exterior_Color: Nominal
- Condition_Level: Ordinal
- Accident_History: Ordinal / Risk Level
- Service_History: Ordinal
- Seller_Type: Nominal
- Seller_Address: Composite / Text Location
- Engine_Spec: Composite Technical Field
- Sale_Status: Binary Outcome
- Salesperson_Favorite_Color: Irrelevant Noise
- Upload_Batch: Operational Metadata
- Photo_URL: Identifier / Irrelevant


### Question 5

In [12]:
print("Condition_Level unique values:", df['Condition_Level'].unique())
print("Service_History unique values:", df['Service_History'].unique())

condition_map = {
    'Fair': 1,
    'Good': 2,
    'Excellent': 3
}
service_map = {
    'None': 0,
    'Partial': 1,
    'Complete': 2
}

df['Condition_Level_Encoded'] = df['Condition_Level'].map(condition_map)
df['Service_History_Encoded'] = df['Service_History'].map(service_map)

print("\nEncoded values:")
print(df[['Condition_Level', 'Condition_Level_Encoded', 'Service_History', 'Service_History_Encoded']].head(5))


Condition_Level unique values: ['Good' 'Very Good' 'Poor' 'Fair' 'Excellent']
Service_History unique values: ['Complete' 'Partial' nan]

Encoded values:
  Condition_Level  ...  Service_History_Encoded
0            Good  ...                      2.0
1       Very Good  ...                      2.0
2       Very Good  ...                      1.0
3       Very Good  ...                      1.0
4       Very Good  ...                      2.0

[5 rows x 4 columns]


### Question 6

In [14]:
print("Sample address:", df['Seller_Address'].iloc[0])

split_addr = df['Seller_Address'].astype(str).str.split(',', expand=True)

df['Street'] = split_addr[0].str.strip()
df['City'] = split_addr[1].str.strip() if 1 in split_addr.columns else None
df['State_Zip'] = split_addr[2].str.strip() if 2 in split_addr.columns else None
if df['State_Zip'] is not None:
    df['State'] = df['State_Zip'].str.split(' ', expand=True)[0]
else:
    df['State'] = None

print("\nSplit Address:")
print(df[['Seller_Address', 'Street', 'City', 'State']].head(3))


Sample address: 836 Maple Rd, Los Angeles, CA

Split Address:
                  Seller_Address          Street         City State
0  836 Maple Rd, Los Angeles, CA    836 Maple Rd  Los Angeles    CA
1       1879 Hill Rd, Denver, CO    1879 Hill Rd       Denver    CO
2      1120 Cedar Ave, Miami, FL  1120 Cedar Ave        Miami    FL


### Question 7

In [16]:
print("Sample Engine_Specs:")
print(df['Engine_Spec'].value_counts().head(5))

def parse_engine_spec(spec):
    if pd.isna(spec):
        return None, None, None
    spec_str = str(spec).strip()
    if 'Electric' in spec_str or 'EV' in spec_str or 'Motor' in spec_str:
        return 0.0, 'Electric', 'Electric Motor'
    
    parts = spec_str.split(' ')
    capacity = None
    unit = 'L'
    fuel_config = ''
    
    if len(parts) > 0:
        val_str = parts[0]
        clean_val = ''.join([c for c in val_str if c.isdigit() or c=='.'])
        try:
            capacity = float(clean_val)
        except:
            capacity = None
            
    if len(parts) > 1:
        fuel_config = ' '.join(parts[1:])
        
    return capacity, unit, fuel_config

df['Engine_Capacity_L'], df['Engine_Unit'], df['Engine_Config'] = zip(*df['Engine_Spec'].apply(parse_engine_spec))
print("\nParsed Engine Spec:")
print(df[['Engine_Spec', 'Engine_Capacity_L', 'Engine_Unit', 'Engine_Config']].head(5))


Sample Engine_Specs:
Engine_Spec
1500 cc | Gasoline    30
3000 cc | Gasoline    26
2500 cc | Gasoline    24
2000 cc | Gasoline    24
3500 cc | Gasoline    22
Name: count, dtype: int64

Parsed Engine Spec:
             Engine_Spec  Engine_Capacity_L Engine_Unit   Engine_Config
0     5000 cc | Gasoline             5000.0           L   cc | Gasoline
1     2500 cc | Gasoline             2500.0           L   cc | Gasoline
2     1800 cc | Gasoline             1800.0           L   cc | Gasoline
3  Single Motor | 150 kW                0.0    Electric  Electric Motor
4       2500 cc | Diesel             2500.0           L     cc | Diesel


### Question 8

In [18]:
df['Mileage_Bin'], bins = pd.qcut(df['Mileage_miles'], q=4, labels=['Low', 'Moderate', 'High', 'Very High'], retbins=True)
print("Data-driven Bin Boundaries for Mileage:")
for i in range(len(bins)-1):
    print(f"- {['Low', 'Moderate', 'High', 'Very High'][i]}: {bins[i]:.1f} to {bins[i+1]:.1f} miles")
print("\nBin distribution:")
print(df['Mileage_Bin'].value_counts())


Data-driven Bin Boundaries for Mileage:
- Low: 11475.0 to 51972.0 miles
- Moderate: 51972.0 to 78463.0 miles
- High: 78463.0 to 117301.0 miles
- Very High: 117301.0 to 650000.0 miles

Bin distribution:
Mileage_Bin
Low          72
Moderate     71
High         71
Very High    71
Name: count, dtype: int64


### Question 9

In [20]:
print(df['Car_Age_Years'].describe())

df['Age_Equal_Width'] = pd.cut(df['Car_Age_Years'], bins=4)
df['Age_Equal_Freq'] = pd.qcut(df['Car_Age_Years'], q=4, duplicates='drop')
df['Age_Domain'] = pd.cut(df['Car_Age_Years'], bins=[0, 3, 7, 12, 100], labels=['New (0-3)', 'Mid-Age (4-7)', 'Older (8-12)', 'Vintage (13+)'])

print("\nEqual Width distribution:")
print(df['Age_Equal_Width'].value_counts().sort_index())
print("\nEqual Frequency distribution:")
print(df['Age_Equal_Freq'].value_counts().sort_index())
print("\nDomain-based distribution:")
print(df['Age_Domain'].value_counts().sort_index())


count    300.000000
mean       8.173333
std        5.179386
min        1.000000
25%        4.000000
50%        7.000000
75%       11.000000
max       32.000000
Name: Car_Age_Years, dtype: float64

Equal Width distribution:
Age_Equal_Width
(0.969, 8.75]    177
(8.75, 16.5]     102
(16.5, 24.25]     20
(24.25, 32.0]      1
Name: count, dtype: int64

Equal Frequency distribution:
Age_Equal_Freq
(0.999, 4.0]    87
(4.0, 7.0]      69
(7.0, 11.0]     71
(11.0, 32.0]    73
Name: count, dtype: int64

Domain-based distribution:
Age_Domain
New (0-3)        60
Mid-Age (4-7)    96
Older (8-12)     85
Vintage (13+)    59
Name: count, dtype: int64


### Question 10

In [22]:
cols_to_check = ['Sale_Price_USD', 'Mileage_miles', 'Days_to_Sell']

for col in cols_to_check:
    series = df[col].dropna()
    
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_iqr = Q1 - 1.5 * IQR
    upper_iqr = Q3 + 1.5 * IQR
    outliers_iqr = df[(df[col] < lower_iqr) | (df[col] > upper_iqr)]
    
    mean = series.mean()
    std = series.std()
    outliers_z = df[np.abs((df[col] - mean) / std) > 3]
    
    print(f"\nOutlier Detection for '{col}':")
    print(f"- IQR boundaries: {lower_iqr:.1f} to {upper_iqr:.1f}")
    print(f"- IQR flagged: {len(outliers_iqr)} records")
    print(f"- Z-score flagged (|Z| > 3): {len(outliers_z)} records")



Outlier Detection for 'Sale_Price_USD':
- IQR boundaries: -12350.0 to 30650.0
- IQR flagged: 6 records
- Z-score flagged (|Z| > 3): 2 records

Outlier Detection for 'Mileage_miles':
- IQR boundaries: -46021.5 to 215294.5
- IQR flagged: 5 records
- Z-score flagged (|Z| > 3): 3 records

Outlier Detection for 'Days_to_Sell':
- IQR boundaries: -15.5 to 60.5
- IQR flagged: 8 records
- Z-score flagged (|Z| > 3): 3 records


### Question 11

In [24]:
skew_cols = ['Sale_Price_USD', 'Mileage_miles', 'Listing_Views', 'Buyer_Inquiries', 'Days_to_Sell']
skewness = df[skew_cols].skew()
print("Skewness measurements:")
for col, val in skewness.items():
    if val > 1:
        suggestion = "Log transform (np.log1p) or Box-Cox (right-skewed)"
    elif val < -1:
        suggestion = "Exponential or power transform (left-skewed)"
    else:
        suggestion = "No transform or Square Root (moderate/no skew)"
    print(f"- {col}: {val:.3f} -> Suggested: {suggestion}")


Skewness measurements:
- Sale_Price_USD: 8.533 -> Suggested: Log transform (np.log1p) or Box-Cox (right-skewed)
- Mileage_miles: 4.102 -> Suggested: Log transform (np.log1p) or Box-Cox (right-skewed)
- Listing_Views: 1.002 -> Suggested: Log transform (np.log1p) or Box-Cox (right-skewed)
- Buyer_Inquiries: 1.604 -> Suggested: Log transform (np.log1p) or Box-Cox (right-skewed)
- Days_to_Sell: 1.068 -> Suggested: Log transform (np.log1p) or Box-Cox (right-skewed)


### Question 12

In [26]:
df_clean = df.copy()

df_clean['Mileage_miles'] = df_clean.groupby('Manufacture_Year')['Mileage_miles'].transform(lambda x: x.fillna(x.median()))
df_clean['Mileage_miles'].fillna(df['Mileage_miles'].median(), inplace=True)

transmission_mode = df_clean['Transmission'].mode()[0]
df_clean['Transmission'].fillna(transmission_mode, inplace=True)

df_clean['Exterior_Color'].fillna('Unknown', inplace=True)

df_clean['Accident_History'].fillna('None', inplace=True)

df_clean['Service_History'].fillna('Unknown', inplace=True)

print("Remaining missing counts in target imputed columns:")
print(df_clean[['Mileage_miles', 'Transmission', 'Exterior_Color', 'Accident_History', 'Service_History']].isnull().sum())


Remaining missing counts in target imputed columns:
Mileage_miles       0
Transmission        0
Exterior_Color      0
Accident_History    0
Service_History     0
dtype: int64


### Question 13

In [28]:
norm_col = 'MinMax_Normalized_Sale_Price'
print("Existing Normalization Range:")
print(f"Min: {df[norm_col].min()}, Max: {df[norm_col].max()}")

prices = df['Sale_Price_USD'].dropna()
Q1 = prices.quantile(0.25)
Q3 = prices.quantile(0.75)
IQR = Q3 - Q1
upper_cap = Q3 + 1.5 * IQR

capped_prices = df['Sale_Price_USD'].clip(upper=upper_cap)

min_p = capped_prices.min()
max_p = capped_prices.max()
df['Recalculated_Normalized_Price'] = (capped_prices - min_p) / (max_p - min_p)

print("\nRecalculated Normalization Range (Capped outliers):")
print(f"Min: {df['Recalculated_Normalized_Price'].min()}, Max: {df['Recalculated_Normalized_Price'].max()}")


Existing Normalization Range:
Min: 0.0, Max: 1.0

Recalculated Normalization Range (Capped outliers):
Min: 0.0, Max: 1.0


### Question 14

In [30]:
drop_cols = [
    'Listing_ID', 'VIN_Code', 'Photo_URL', 
    'Salesperson_Favorite_Color', 'Random_Noise_Code', 
    'Upload_Batch', 'MinMax_Normalized_Sale_Price', 
    'Price_per_1000_Miles'
]
print("Suggested columns to drop:")
for c in drop_cols:
    print(f"- {c}")


Suggested columns to drop:
- Listing_ID
- VIN_Code
- Photo_URL
- Salesperson_Favorite_Color
- Random_Noise_Code
- Upload_Batch
- MinMax_Normalized_Sale_Price
- Price_per_1000_Miles


### Question 15

In [32]:
print("Cross tabulation of Condition_Level and Condition_Score:")
print(pd.crosstab(df['Condition_Level'], df['Condition_Score']))

age_year_corr = df['Manufacture_Year'].corr(df['Car_Age_Years'])
print(f"\nCorrelation between Manufacture_Year and Car_Age_Years: {age_year_corr:.4f}")


Cross tabulation of Condition_Level and Condition_Score:
Condition_Score   1   2   3    4   5
Condition_Level                     
Excellent         0   0   0    0  48
Fair              0  36   0    0   0
Good              0   0  93    0   0
Poor             11   0   0    0   0
Very Good         0   0   0  112   0

Correlation between Manufacture_Year and Car_Age_Years: -1.0000


### Question 16

In [34]:
temp = df[['Brand', 'Sale_Price_USD', 'Car_Age_Years', 'Mileage_miles']].dropna()

X = temp[['Car_Age_Years', 'Mileage_miles']].values
X = np.hstack([np.ones((X.shape[0], 1)), X])
y = temp['Sale_Price_USD'].values

beta, residuals_sum, rank, s = np.linalg.lstsq(X, y, rcond=None)

predictions = X.dot(beta)
temp['Price_Residual'] = y - predictions

brand_retention = temp.groupby('Brand')['Price_Residual'].mean().sort_values(ascending=False)
print("Brand Value Retention (Average Price Residual after controlling for Age and Mileage):")
print(brand_retention)


Brand Value Retention (Average Price Residual after controlling for Age and Mileage):
Brand
Kia              8741.723952
BMW              5676.731638
Mercedes-Benz    1461.786804
Nissan           -712.123418
Honda            -712.857780
Toyota           -924.511306
Hyundai         -1255.817327
Chevrolet       -1408.133506
Ford            -2122.370185
Volkswagen      -2448.630047
Name: Price_Residual, dtype: float64


### Question 17

In [36]:
dealer_comparison = df.groupby('Seller_Type')[['Car_Age_Years', 'Condition_Score', 'Sale_Price_USD']].mean()
print("Comparison by Seller Type:")
print(dealer_comparison)


Comparison by Seller Type:
                  Car_Age_Years  Condition_Score  Sale_Price_USD
Seller_Type                                                     
Certified Dealer       7.610390         3.519481    10910.769231
Dealer                 8.393939         3.522727    10833.035714
Private                8.329670         3.450549     9913.924051


### Question 18

In [38]:
df['Inquiries_per_100_Views'] = (df['Buyer_Inquiries'] / df['Listing_Views']) * 100
attention_unsold = df[df['Sale_Status'] == 'Unsold'].sort_values(by='Listing_Views', ascending=False)
print("Top Unsold Listings with highest views:")
print(attention_unsold[['Listing_ID', 'Brand', 'Model', 'Asking_Price_USD', 'Listing_Views', 'Buyer_Inquiries', 'Inquiries_per_100_Views']].head(5))


Top Unsold Listings with highest views:
    Listing_ID          Brand  ... Buyer_Inquiries  Inquiries_per_100_Views
193   CAR-0194         Toyota  ...               5                 0.854701
19    CAR-0020  Mercedes-Benz  ...               9                 1.707780
110   CAR-0111        Hyundai  ...               5                 1.149425
98    CAR-0099            Kia  ...               6                 1.948052
123   CAR-0124            BMW  ...               4                 1.311475

[5 rows x 7 columns]


### Question 19

In [40]:
df['Discount_Percentage'] = (df['Asking_Price_USD'] - df['Sale_Price_USD']) / df['Asking_Price_USD'] * 100
print("Average Discount Percentage:", df['Discount_Percentage'].mean())

numeric_features = ['Mileage_miles', 'Car_Age_Years', 'Condition_Score', 'Listing_Views']
corrs = df[numeric_features].corrwith(df['Discount_Percentage'])
print("\nCorrelation of Discount Percentage with characteristics:")
print(corrs)


Average Discount Percentage: 9.473422714918632

Correlation of Discount Percentage with characteristics:
Mileage_miles     -0.108434
Car_Age_Years     -0.109789
Condition_Score    0.043589
Listing_Views     -0.084113
dtype: float64


### Question 20

In [42]:
sold_cars = df[df['Sale_Status'] == 'Sold']
corrs_speed = sold_cars[numeric_features].corrwith(sold_cars['Days_to_Sell'])
print("Correlation of Days to Sell with characteristics (Sold cars only):")
print(corrs_speed)


Correlation of Days to Sell with characteristics (Sold cars only):
Mileage_miles     -0.096850
Car_Age_Years     -0.114634
Condition_Score    0.011619
Listing_Views      0.064685
dtype: float64


### Question 21

In [44]:
luxury_brands = ['BMW', 'Audi', 'Mercedes-Benz', 'Lexus', 'Porsche']
df['Is_Luxury'] = df['Brand'].isin(luxury_brands).astype(int)

temp_interaction = df[['Sale_Price_USD', 'Mileage_miles', 'Is_Luxury']].dropna()
temp_interaction['Mileage_x_Luxury'] = temp_interaction['Mileage_miles'] * temp_interaction['Is_Luxury']

X_int = temp_interaction[['Mileage_miles', 'Is_Luxury', 'Mileage_x_Luxury']].values
X_int = np.hstack([np.ones((X_int.shape[0], 1)), X_int])
y_int = temp_interaction['Sale_Price_USD'].values

beta_int, _, _, _ = np.linalg.lstsq(X_int, y_int, rcond=None)
print("Regression coefficients (Intercept, Mileage, Is_Luxury, Mileage * Is_Luxury):")
features = ['Intercept', 'Mileage_miles', 'Is_Luxury', 'Mileage_x_Luxury']
for feat, coef in zip(features, beta_int):
    print(f"- {feat}: {coef:.4f}")


Regression coefficients (Intercept, Mileage, Is_Luxury, Mileage * Is_Luxury):
- Intercept: 14391.1906
- Mileage_miles: -0.0479
- Is_Luxury: 13412.3359
- Mileage_x_Luxury: -0.1122


### Question 22

In [46]:
anomalies = df[(df['Car_Age_Years'] > 8) & (df['Mileage_miles'] < 15000) & (df['Sale_Price_USD'] > df['Sale_Price_USD'].median())]
print("Identified anomalies (Low Mileage + Old Age + High Price):")
print(anomalies[['Listing_ID', 'Brand', 'Model', 'Car_Age_Years', 'Mileage_miles', 'Sale_Price_USD', 'Condition_Level']])


Identified anomalies (Low Mileage + Old Age + High Price):
Empty DataFrame
Columns: [Listing_ID, Brand, Model, Car_Age_Years, Mileage_miles, Sale_Price_USD, Condition_Level]
Index: []


### Question 23

In [48]:
overall_corr = df['Mileage_miles'].corr(df['Sale_Price_USD'])
print(f"Overall Mileage vs Sale Price correlation: {overall_corr:.4f}")

print("\nCorrelation by Vehicle Type:")
print(df.groupby('Vehicle_Type')[['Mileage_miles', 'Sale_Price_USD']].corr().iloc[0::2,-1].reset_index(level=1, drop=True))


Overall Mileage vs Sale Price correlation: -0.3015

Correlation by Vehicle Type:
Vehicle_Type
Hatchback    0.043890
SUV         -0.222757
Sedan       -0.791997
Truck       -0.865522
Name: Sale_Price_USD, dtype: float64


### Question 24

In [50]:
print("Regression Dataset size (Sold only):", df[df['Sale_Status'] == 'Sold'].shape)
print("Classification Dataset size (All records):", df.shape)


Regression Dataset size (Sold only): (256, 50)
Classification Dataset size (All records): (300, 50)


### Question 25

In [52]:
df['Expected_Margin'] = df['Asking_Price_USD'] - df['Sale_Price_USD']
df['Prob_Sale'] = df['Sale_Status'].map({'Sold': 1.0, 'Unsold': 0.5})
df['Speed_Factor'] = 1 / (df['Days_to_Sell'].fillna(df['Days_to_Sell'].median()) + 1)

df['Deal_Score'] = df['Expected_Margin'].fillna(0) * df['Prob_Sale'] * df['Speed_Factor']
top_20 = df.sort_values(by='Deal_Score', ascending=False).head(20)

print("Top 5 recommended vehicles to purchase:")
print(top_20[['Listing_ID', 'Brand', 'Model', 'Asking_Price_USD', 'Sale_Price_USD', 'Deal_Score']].head(5))


Top 5 recommended vehicles to purchase:
    Listing_ID          Brand  ... Sale_Price_USD  Deal_Score
12    CAR-0013            BMW  ...        17800.0  966.666667
124   CAR-0125            BMW  ...        50400.0  455.555556
200   CAR-0201            Kia  ...       173900.0  396.428571
273   CAR-0274            BMW  ...        33300.0  333.333333
237   CAR-0238  Mercedes-Benz  ...        12900.0  328.571429

[5 rows x 6 columns]
